# 15 — M2 Gate

**Autoridad formal de cierre de M2.** Consume evidencia persistida por 10–14 y decide computablemente si la capa Silver puede congelarse antes de comenzar Gold.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.silver import load_silver_spec

paths = ProjectPaths.discover(); paths.ensure_runtime_dirs()
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")
{"schema_version": SILVER_SPEC.schema_version, "reports": str(paths.reports / "silver")}

In [ ]:
from argos.evaluation.m2_gate import evaluate_m2_gate, persist_m2_gate

report, decision = evaluate_m2_gate(
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    schema_version=SILVER_SPEC.schema_version,
)
report

In [ ]:
report_path, decision_path = persist_m2_gate(
    report=report,
    decision=decision,
    reports_root=paths.reports,
    decision_path=paths.docs / "decisions" / "M2_GATE.md",
)

print("M2 ✅ CLOSED" if decision.closed else "M2 ❌ OPEN")
print(f"Blocking checks: {decision.blocking_passed}/{decision.blocking_total}")
if decision.failed_checks:
    print("Failed:", ", ".join(decision.failed_checks))
print("Gate report:", report_path)
print("Decision artifact:", decision_path)

assert decision.closed, (
    "M2 remains OPEN. Resolve the failed checks above and rerun the affected notebooks before starting Gold."
)

## Después del cierre

Si el gate imprime `M2 ✅ CLOSED`, congelar `docs/decisions/M2_GATE.md`, los manifests y los reportes Silver. El siguiente milestone es Gold, empezando por target engineering y diseño del split temporal. **No entrenar modelos directamente sobre Silver.**